In [ ]:
import pandas as pd
import numpy as np

import gspread
import panel as pn
pn.extension('tabulator')
import hvplot.pandas
import holoviews as hv
hv.extension('bokeh')
#pn.extension(theme='dark')

In [ ]:
#after renaming the json file
gc = gspread.service_account(filename="service_account.json")
sh = gc.open('bank_statement_april')
ws=sh.worksheet('Sheet1')
df = pd.DataFrame(ws.get_all_records()) 
df.head()


In [ ]:
#cleaning the dataframe
df = df[['Date', 'Narration', 'Amount']]
df['Narration']=df['Narration'].map(str.lower)
df = df.rename(columns={'Narration':'Description'})
#add a category column
df['Category']='unassigned'
df.head()

In [ ]:
""" 
Categories defined as:
1. Friend
2. Transport
3. Self Care
4. Grocery
5. Dine out
"""

In [ ]:
#df['Category'] = 'unassigned'

# Assign categories based on 'Narration' using np.where
df['Category'] = np.where(df['Description'].str.contains('Utilities Payment', case=False), 'Utilities', df['Category'])
df['Category'] = np.where(df['Description'].str.contains('Salary Deposit', case=False), 'Salary', df['Category'])
df['Category'] = np.where(df['Description'].str.contains('ATM Withdrawal', case=False), 'ATM', df['Category'])
df['Category'] = np.where(df['Description'].str.contains('Online Shopping', case=False), 'Shopping', df['Category'])
df['Category'] = np.where(df['Description'].str.contains('Grocery Store', case=False), 'Grocery', df['Category'])

# Set 'Date' to datetime format
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')  # Using 'coerce' to handle invalid date formats

# Add 'Month' and 'Year' columns
df['Month'] = df['Date'].dt.month
df['Year'] = df['Date'].dt.year

# Display the first 200 rows of the DataFrame
pd.options.display.max_rows = 999
df.head(200)


In [ ]:
#find the transations not assigned a category yet
unassigned = df.loc[df['Category']=='unassigned']
unassigned


In [ ]:
latest_month=df['Month'].max()
latest_year=df['Year'].max()
#filter the df to hold only recent expenses

latest_expense=df[((df['Month']==latest_month)|(df['Month']==latest_month-1))&(df['Year']==latest_year)]

In [ ]:
#FOR ELEMENT 1
last_month_expenses=latest_expense.groupby('Category')['Amount'].sum().reset_index()
last_month_expenses['Amount']=last_month_expenses['Amount'].astype('str')
last_month_expenses['Amount']=last_month_expenses['Amount'].str.replace('-',' ')
last_month_expenses['Amount']=last_month_expenses['Amount'].astype('float')
#after getting the absolute figures, get all those which have assigned categories

last_month_expenses=last_month_expenses[last_month_expenses['Category'].str.contains('unassigned')==False]
last_month_expenses=last_month_expenses.sort_values(by='Amount', ascending=False)
last_month_expenses['Amount']=last_month_expenses['Amount'].round().astype(int)

last_month_expenses

In [ ]:
last_month_expenses_tot=last_month_expenses['Amount'].sum()
float(last_month_expenses_tot)

In [ ]:
def calc_diff(event):
    income=float(income_widget.value)
    recurring_expenses=float(recurring_expenses_widget.value)
    monthly_expenses=float(monthly_expenses_widget.value)
    diff=income-recurring_expenses-monthly_expenses
    difference_widget.value=str(diff)

income_widget=pn.widgets.TextInput(name='Income',value="0")
recurring_expenses_widget=pn.widgets.TextInput(name='Recurring Expenses',value='0')
monthly_expenses_widget=pn.widgets.TextInput(name='Non-Recurring Expenses', value=str(last_month_expenses_tot))
difference_widget=pn.widgets.TextInput(name="Last Month's Savings", value="0")

income_widget.param.watch(calc_diff,"value")
recurring_expenses_widget.param.watch(calc_diff,"value")
monthly_expenses_widget.param.watch(calc_diff,"value")
pn.Row(income_widget,recurring_expenses_widget,monthly_expenses_widget,difference_widget).show()


In [ ]:
#creating the bar chart to visualize all the categories
last_month_expenses_chart=last_month_expenses.hvplot.bar(
    x='Category',
    y='Amount',
    height=250,
    width=850,
    title='Last Month Expenses',
    ylim=(0,500)
)
last_month_expenses_chart



In [ ]:
#monthly expenses trend bar chart
df['Date']=pd.to_datetime(df['Date'])
df['Month-Year']=df['Date'].dt.to_period('M')
monthly_expenses_by_cat =df.groupby(['Month-Year','Category'])['Amount'].sum().reset_index()

monthly_expenses_by_cat['Amount']=monthly_expenses_by_cat['Amount'].astype('str')
monthly_expenses_by_cat['Amount']=monthly_expenses_by_cat['Amount'].str.replace('-',' ')
monthly_expenses_by_cat['Amount']=monthly_expenses_by_cat['Amount'].astype('float')
monthly_expenses_by_cat=monthly_expenses_by_cat[monthly_expenses_by_cat['Category'].str.contains("unassigned")==False]
monthly_expenses_by_cat=monthly_expenses_by_cat.sort_values(by='Amount', ascending=False)
monthly_expenses_by_cat['Amount']=monthly_expenses_by_cat['Amount'].round().astype(int)
monthly_expenses_by_cat['Month-Year']=monthly_expenses_by_cat['Month-Year'].astype('str')
monthly_expenses_by_cat=monthly_expenses_by_cat.rename(columns={'Amount': 'Amount '})
monthly_expenses_by_cat


In [ ]:

    #let's define the panel widget
select_cat1=pn.widgets.Select(name='Select Category', options=['All']+list(monthly_expenses_by_cat['Category'].unique()))
select_cat1

In [ ]:
def plot_expenses(category):
    if category=='All':
        plot_df=monthly_expenses_by_cat.groupby('Month-Year').sum()
    else:
        plot_df=monthly_expenses_by_cat[monthly_expenses_by_cat['Category']==category].groupby('Month-Year').sum()
    plot=plot_df.hvplot.bar(x='Month-Year', y='Amount ')
    return plot

#define a callback for when the value changes after you pick something else
@pn.depends(select_cat1.param.value)
def update_plot(category):
    plot=plot_expenses(category)
    return plot
monthly_expenses_by_cat_chart=pn.Row(select_cat1, update_plot)
monthly_expenses_by_cat_chart[1].width=600
monthly_expenses_by_cat_chart

In [ ]:
#summary table

df=df[['Date','Category','Description','Amount']]
df['Amount']=df['Amount'].astype('str')
df['Amount']=df['Amount'].str.replace('-',' ')
df['Amount']=df['Amount'].astype('float')
df=df[df['Category'].str.contains('unassigned')==False]
df['Amount']=df['Amount'].round().astype(int)
df

In [ ]:
#filter df based on category
def filter_df(category):
    if category=='All':
        return df
    return df[df['Category']==category]

summary_table=pn.widgets.DataFrame(filter_df('All'), height=500, width=650)
def update_summary(event):
    summary_table.value=filter_df(event.new)
select_cat1.param.watch(update_summary, 'value')
summary_table

In [ ]:
#finally put everything into our dashboard
template = pn.template.FastListTemplate(
    title="Anukriti's Personal Finances Summary",
    sidebar=pn.Column(
        pn.pane.Markdown("*Don't go broke trying to look rich...*"),
        pn.pane.PNG('image.png', sizing_mode='scale_both', max_width=300, height=200),
        pn.pane.Markdown(" "),
        pn.pane.Markdown(" "), 
        select_cat1
    ),
    main=[
        # First row with the widgets
        pn.Row(
            income_widget, 
            recurring_expenses_widget, 
            monthly_expenses_widget, 
            difference_widget, 
            width=950,  # Adjust the width as needed for the layout
            sizing_mode='stretch_width'  # Ensure it stretches across the available space
        ),
        
        # Second row with last month expenses chart
        pn.Row(
            last_month_expenses_chart, 
            height=240,
            sizing_mode='stretch_width'  # Ensure chart stretches to the available space
        ),
        
        # Third row with monthly expenses by category chart in GridBox
        pn.GridBox(
            monthly_expenses_by_cat_chart[1],  # Assuming monthly_expenses_by_cat_chart[1] is a valid chart
            ncols=2,  # Two columns
            width=600,  # Set an appropriate width for the grid box
            height=400,  # Set height
            align='center',  # Center align the content
            sizing_mode='stretch_width'  # Make sure it stretches to available width
        ),
        
        # Fourth row with summary table in GridBox
        pn.GridBox(
            summary_table, 
            sizing_mode='scale_width'  # Scale width to ensure the table fits
        )
    ]
)

# Show the dashboard
template.show()